# Local NBA historical access test

**Question:** Can this computer retrieve and validate player and team history through the native `nba_api` client?

Use the repository's `.venv` Python 3.12 kernel. Run cells in order. Defaults check synthetic fixtures without network requests. Live mode makes at most two requests, 30 seconds each, with no retries and a three second gap.

The [maintainers' example](https://github.com/swar/nba_api/blob/master/docs/nba_api/stats/examples.md) sends requests by constructing an endpoint, then reads a DataFrame. This revision uses `LeagueGameLog(...)` and `league_game_log.get_data_frame()` directly through the installed package. The documented `headers` option preserves Baseline's identity instead of the package's browser identity. No proxy is added or rotated.

The earlier urllib probe timed out for both history types. Its evidence remains in `artifacts/notebook-history/`; this test writes a separate native-client archive. This is not a cloud, schedule, or full-season backfill test.


## Latest observed result

**October 15, 2026, 21:09 UTC: stock client access worked.** Player sample: HTTP 200, 636 validated rows. Team sample: HTTP 200, 60 validated rows. Python 3.12.14, nba_api 1.11.4.

Open the [executed notebook](artifacts/notebook-revisions/20261015T210943Z-stock-executed.ipynb) for the actual outputs. Diagnostic evidence is in `artifacts/notebook-history-stock/ecb4ebeb757f4e459465e0d71c47d170/`. These files are local and ignored by Git.

The earlier honest-header section is retained for comparison. Leave it disabled. The **Stock client diagnostic** section below exercises the approved configuration. This editable notebook defaults to offline checks; no rerun is needed to inspect the saved result. Full collection policy and completeness remain separate review gates.


In [ ]:
import json
import os
import subprocess
import sys
from dataclasses import asdict
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import pandas as pd
from IPython.display import display

from hoop.domains.nba.sources.native_history import HEADERS, NBAApiHistoryTransport
from hoop.domains.nba.sources.policy import Policy, ProbeRunner, SystemClock
from hoop.domains.nba.sources.probe import local_requests
from hoop.domains.nba.sources.validate import validate
from hoop.platform.provenance import Archive, digest, utc

ROOT = next(
    folder
    for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "pyproject.toml").is_file() and (folder / "src/hoop").is_dir()
)
assert sys.version_info[:2] == (3, 12), "Select the project's Python 3.12 kernel."
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Select this repository's .venv."
print(f"Python: {sys.version.split()[0]} | nba_api: {version('nba-api')}")
print(f"Interpreter: {sys.executable}")

## Inputs

The sample is April 10 to 12, 2026, from the 2025-26 regular season. Player and team requests are checked independently. Keep the date window small.

To make live requests, set **both switches below to `True`** on your own computer. Run live mode once and inspect the evidence. A denied source stops; do not change identity or repeatedly retry it.

In [ ]:
RUN_LIVE = False
CONFIRM_LOCAL_MACHINE = False

config = json.loads((ROOT / "configs/sources/local.json").read_text())
history_requests = local_requests({**config, "team_ids": []})
assert len(history_requests) == 2
assert {request.context["side"] for request in history_requests} == {"P", "T"}
assert all(request.capability == "history" for request in history_requests)
print(f"Season: {config['history_season']}")
print(f"Dates: {config['history_date_from']} to {config['history_date_to']}")
print(f"Mode: {'LIVE LOCAL' if RUN_LIVE else 'OFFLINE FIXTURES'}")

## Fixture check

These synthetic files check the parser and installation. Their success is **not evidence of NBA access**.

In [ ]:
fixture_rows = []
for request in history_requests:
    label = "player" if request.context["side"] == "P" else "team"
    body = (ROOT / f"tests/fixtures/nba/history-{label}.json").read_bytes()
    summary = validate(body, request, datetime.now(UTC))
    fixture_rows.append(
        {
            "history": label,
            "evidence": "synthetic fixture",
            "parser": "passed",
            "rows": summary["record_count"],
        }
    )
fixture_summary = pd.DataFrame(fixture_rows)
display(fixture_summary)

## Live local probe

The shared helper calls the native endpoint constructor with the same season, dates, and player/team parameters. It uses a temporary Requests session only to retain response bytes before the package parser runs, then restores the previous session. Redirects are stopped; no retries are configured. The client's normal response buffering remains, so this diagnostic is not a production download limiter. Only responses within the 8 MiB validation limit can pass.

Raw payloads and manifests go to `artifacts/notebook-history-native/<run-id>/`. A timeout is not proof of denial. HTTP 403 or an HTML challenge stops the host. No API key is needed. Leave both switches false for an offline run; set both true to request this bounded local test.


In [ ]:
live_runner = None
archive_path = None
if not RUN_LIVE:
    print("Live probe skipped. Both switches must be True to test actual access.")
else:
    if not CONFIRM_LOCAL_MACHINE:
        raise RuntimeError("Confirm this is your own local computer before running live mode.")
    if (
        os.environ.get("GITHUB_ACTIONS") == "true"
        or os.environ.get("CI", "").lower() == "true"
        or "CODEX_PROXY_CERT" in os.environ
    ):
        raise RuntimeError("Historical stats requests are forbidden from cloud/CI runners.")

    policy = Policy(min_gap=3, timeout=30, retries=0, max_requests=2, max_seconds=75)
    run_id = uuid4().hex
    archive_path = ROOT / "artifacts/notebook-history-native" / run_id
    archive = Archive(archive_path)
    native_transport = NBAApiHistoryTransport(archive, confirm_local_machine=CONFIRM_LOCAL_MACHINE)
    live_runner = ProbeRunner(archive, native_transport, SystemClock(), policy, run_id)
    identity = {
        "run_id": run_id,
        "environment": "local",
        "evidence": "live network",
        "client": "nba_api.LeagueGameLog + get_data_frame",
        "headers": HEADERS,
        "native_helper_sha256": digest(
            (ROOT / "src/hoop/domains/nba/sources/native_history.py").read_bytes()
        ),
        "started_at": utc(datetime.now(UTC)),
        "code_commit": subprocess.check_output(
            ["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True
        ).strip(),
        "nba_api_version": version("nba-api"),
        "dependency_lock_sha256": digest((ROOT / "uv.lock").read_bytes()),
        "notebook_sha256": digest((ROOT / "test.ipynb").read_bytes()),
        "policy": asdict(policy),
        "requests": [
            {"url": request.url, "context": request.context} for request in history_requests
        ],
    }
    (archive_path / "identity.json").write_text(json.dumps(identity, indent=2), encoding="utf-8")
    for request in history_requests:
        label = "player" if request.context["side"] == "P" else "team"
        print(f"Checking {label} history (at most 30 seconds)...", flush=True)
        observation = live_runner.fetch(request, native_transport.validate)
        print(f"{label}: {observation['outcome']} | {observation['reason'] or 'validated sample'}")
    archive.verify()
    print(f"Evidence: {archive_path}")
    print("Client: native nba_api LeagueGameLog; identity: Baseline")

## Inspect and interpret

Both live history types must pass. One successful type cannot conceal a failure in the other. An empty response, schema failure, or timeout is not usable history. Unknown values remain unknown; reported whole minutes do not prove second-level precision.

In [ ]:
live_access_passed = False
if live_runner is not None:
    live_summary = pd.DataFrame(
        [
            {
                "history": "player" if row["request_context"]["side"] == "P" else "team",
                "evidence": "live network",
                "outcome": row["outcome"],
                "HTTP": row["http_status"],
                "rows": row["record_count"],
                "reason": row["reason"],
            }
            for row in live_runner.observations
        ]
    )
    display(live_summary)
    live_access_passed = (
        len(live_runner.observations) == 2
        and {row["request_context"]["side"] for row in live_runner.observations} == {"P", "T"}
        and all(row["outcome"] == "verified_sample" for row in live_runner.observations)
    )
    print(f"Both historical samples verified: {live_access_passed}")
    print(f"Requests used: {live_runner.requests}/{live_runner.policy.max_requests}")
else:
    print("Actual NBA access has not been tested by this notebook run.")

## Next step

If both live samples pass, review their manifests before accepting the historical route for DATA-001. This small sample does not establish full-season completeness, current rosters, cloud access, or historical feature availability.

If either fails, keep historical collection on hold and share the summary and run path. Do not repeatedly rerun a denied request. Clear live outputs before committing this notebook; keep raw data in the ignored artifact folder.

## Stock client diagnostic: purpose and inputs

The owner approved this **local diagnostic only** on October 15, 2026. It tests the installed package's unchanged headers, including its Chrome identity and NBA Referer. This does not change the production source policy. See the [maintainer's header fix](https://github.com/swar/nba_api/pull/636) and [endpoint documentation](https://github.com/swar/nba_api/blob/master/docs/nba_api/stats/examples.md).

Restart the kernel. Leave the earlier `RUN_LIVE = False`. Set only the two switches below to `True` to test stock access. At most two endpoint calls use the same small regular-season sample, with a 20 second connect/read timeout, no configured retries, and a three second gap. Stop after the first failure. Requests uses its normal redirect behavior; the timeout is not a total wall-clock deadline.

No browser cookies are copied, no headers or sessions are replaced, and no proxy is added. A fresh session and absence of configured proxies are checked before calling the endpoint. Responses exposed by the native client are decoded text, **not original wire bytes**. This section is diagnostic evidence, not the production archive.

In [ ]:
RUN_STOCK_DEFAULTS = False
STOCK_LOCAL_CONFIRMED = False

In [ ]:
from time import monotonic, sleep

import nba_api.library.http as native_http
import requests
from nba_api.stats.endpoints.leaguegamelog import LeagueGameLog
from nba_api.stats.library.http import NBAStatsHTTP

stock_results = []
stock_archive = None
stock_access_passed = False

if not RUN_STOCK_DEFAULTS:
    print("Stock client: skipped. Defaults remain offline.")
else:
    if not STOCK_LOCAL_CONFIRMED:
        raise RuntimeError("Confirm this is your own local computer for the approved diagnostic.")
    if any(
        os.environ.get(key, "").lower() == "true" for key in ["CI", "GITHUB_ACTIONS"]
    ) or os.environ.get("CODEX_PROXY_CERT"):
        raise RuntimeError("Stock diagnostic is forbidden from cloud/CI.")
    if NBAStatsHTTP._session is not None:
        raise RuntimeError("Restart the kernel: stock diagnostic requires a fresh native session.")
    if (
        native_http.PROXY
        or native_http.DEBUG
        or native_http.DEBUG_STORAGE
        or requests.utils.get_environ_proxies(history_requests[0].url)
    ):
        raise RuntimeError("Remove configured proxies/debug mode before this diagnostic.")

    stock_archive = ROOT / "artifacts/notebook-history-stock" / uuid4().hex
    stock_archive.mkdir(parents=True)
    stock_identity = {
        "started_at": utc(datetime.now(UTC)),
        "mode": "owner-approved local stock-client diagnostic only",
        "python": sys.version.split()[0],
        "nba_api": version("nba-api"),
        "headers": dict(NBAStatsHTTP.headers),
        "notebook_sha256": digest((ROOT / "test.ipynb").read_bytes()),
        "lock_sha256": digest((ROOT / "uv.lock").read_bytes()),
        "git_head": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=ROOT, text=True
        ).strip(),
        "maximum_endpoint_calls": 2,
        "timeout_seconds": 20,
        "retries": 0,
        "requests": [{"url": item.url, "context": item.context} for item in history_requests],
        "body_format": "native-client decoded response text re-encoded as UTF-8, not wire bytes",
    }
    (stock_archive / "identity.json").write_text(
        json.dumps(stock_identity, indent=2), encoding="utf-8"
    )
    for item in history_requests:
        if stock_results:
            sleep(3)
        endpoint = LeagueGameLog(
            season=item.context["season"],
            player_or_team_abbreviation=item.context["side"],
            season_type_all_star="Regular Season",
            date_from_nullable=item.context["date_from"],
            date_to_nullable=item.context["date_to"],
            timeout=20,
            get_request=False,
        )
        # get_request=False preserves the endpoint object if native parsing fails.
        # get_request() sends the same normal request with unchanged client defaults.
        row = {
            "side": item.context["side"],
            "started_at": utc(datetime.now(UTC)),
            "outcome": "failed",
            "http_status": None,
            "rows": 0,
            "error": None,
        }
        started = monotonic()
        try:
            endpoint.get_request()
            frame = endpoint.league_game_log.get_data_frame()
            response = endpoint.nba_response
            body = response.get_response().encode("utf-8")
            row["http_status"] = response._status_code
            if row["http_status"] != 200:
                raise ValueError("Non-200 response is not accepted data")
            checked = validate(body, item, datetime.now(UTC))
            row.update(outcome="verified_sample", rows=checked["record_count"])
            display(
                frame[
                    [
                        name
                        for name in [
                            "GAME_DATE",
                            "TEAM_ID",
                            "PLAYER_ID",
                            "MIN",
                            "PTS",
                            "REB",
                            "AST",
                            "FG3M",
                        ]
                        if name in frame
                    ]
                ].head(3)
            )
        except Exception as exc:
            row["error"] = type(exc).__name__
        finally:
            row["elapsed_seconds"] = round(monotonic() - started, 3)
            row["finished_at"] = utc(datetime.now(UTC))
            if endpoint.nba_response is not None:
                body = endpoint.nba_response.get_response().encode("utf-8")
                row["http_status"] = endpoint.nba_response._status_code
                row["response_sha256"] = digest(body)
                row["response_file"] = f"{item.context['side']}-response-text.txt"
                (stock_archive / row["response_file"]).write_bytes(body)
            stock_results.append(row)
            (stock_archive / "results.json").write_text(
                json.dumps(stock_results, indent=2), encoding="utf-8"
            )
        if row["outcome"] != "verified_sample":
            break
    print("Stock diagnostic evidence:", stock_archive)

In [ ]:
if stock_results:
    display(
        pd.DataFrame(stock_results)[
            ["side", "outcome", "http_status", "rows", "error", "elapsed_seconds"]
        ]
    )
    stock_access_passed = (
        len(stock_results) == 2
        and {row["side"] for row in stock_results} == {"P", "T"}
        and all(row["outcome"] == "verified_sample" for row in stock_results)
    )
    print("Both historical samples validated:", stock_access_passed)
    if not stock_access_passed:
        print(
            "Access not demonstrated. Unattempted samples remain unverified, not failed or empty."
        )
else:
    print("No live stock-client evidence in this kernel.")

## Interpretation

- `verified_sample` means a nonempty HTTP 200 response passed the existing season, dates, identity, minutes, and counting-stat checks.
- `ReadTimeout` with no HTTP status means no HTTP response reached the native client before its timeout. It does not establish a permanent block or identify who caused the delay.
- A parser error with an HTTP status means a response arrived but was not accepted. Inspect the saved text privately. Do not replay cookies or retry a denial.
- Both samples must pass for this historical route to be considered for further validation. Full-season completeness, roster reconstruction, source permissions, and the separate cloud schedule gate still require evidence.

Keep the executed result locally under `artifacts/`. Reset all four live switches and clear outputs before committing this notebook.

## Historical dataset assessment

This offline check uses invented CSV fixtures and the shared source assessor. It checks pinned hashes, IDs, targets, joins, participation, and a declared game inventory. It creates no normalized seed, repairs, network requests, or forecasts. Real-source acceptance remains separate.


In [ ]:
from hoop.domains.nba.sources.bulk_history import assess_snapshot

fixture_root = ROOT / "tests/fixtures/nba/eoin-history"
dataset_summary = assess_snapshot(
    fixture_root,
    json.loads((fixture_root / "manifest.json").read_text()),
    "2025-26",
    json.loads((fixture_root / "inventory.json").read_text()),
)
display(
    {
        key: dataset_summary[key]
        for key in [
            "evidence_kind",
            "status",
            "valid_game_count",
            "valid_team_rows",
            "source_accepted",
        ]
    }
)

**Interpretation:** Passing the fixture confirms the assessor works on this example. It does not accept Eoin history. A real assessment must use the pinned download manifest; missing expected-game evidence stays unknown. Repairs are proposed records with reasons and provenance, never edits to the raw CSVs. See `docs/PRD-data-source-amendment.md`.
